<a href="https://colab.research.google.com/github/A-s-n55555/Sports_vs_Politics/blob/main/M25CSE004_Sport_vs_Politics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

file_path = "data.csv"

df = pd.read_csv(file_path)

print(df.head())
print(df.columns)



                                                text  label
0  Captains lining up for Aid match  Ireland's Br...      1
1  Ferguson rues failure to cut gap  Boss Sir Ale...      1
2  Blunkett sorry over murder plan  David Blunket...      0
3  EU rules 'won't stop UK spending'  The shape o...      0
4  Tories attack EU asylum moves  David Blunkett ...      0
Index(['text', 'label'], dtype='object')


In [ ]:
print(df.columns)


Index(['text', 'label'], dtype='object')


In [ ]:
df["label"].value_counts()


,count
label,
1,511
0,417


In [ ]:
df_sp = df.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
df_sp.head()


,category,filename,title,content,label,text
0,sport,339.txt,Captains lining up for Aid match,Ireland's Brian O'Driscoll is one of four Six...,1,Captains lining up for Aid match Ireland's Br...
1,sport,252.txt,Ferguson rues failure to cut gap,Boss Sir Alex Ferguson was left ruing Manches...,1,Ferguson rues failure to cut gap Boss Sir Ale...
2,politics,031.txt,Blunkett sorry over murder plan,David Blunkett has apologised to MPs after th...,0,Blunkett sorry over murder plan David Blunket...
3,politics,389.txt,EU rules 'won't stop UK spending',The shape of the UK's economy In graphics Bu...,0,EU rules 'won't stop UK spending' The shape o...
4,politics,193.txt,Tories attack EU asylum moves,David Blunkett has been accused of using the ...,0,Tories attack EU asylum moves David Blunkett ...


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split


X = df_sp["text"]
y = df_sp["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(len(X_train), len(X_test))


742 186


Model 1: TF-IDF + Linear SVM:

In [ ]:
class LinearSVC:
    def __init__(self, lr=0.001, C=1.0, epochs=1000):
        self.lr = lr
        self.C = C
        self.epochs = epochs
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.epochs):
            for i in range(n_samples):
                condition = y[i] * (np.dot(X[i], self.w) + self.b)

                if condition >= 1:
                    self.w -= self.lr * self.w
                else:
                    self.w -= self.lr * (self.w - self.C * y[i] * X[i])
                    self.b -= self.lr * (-self.C * y[i])

    def predict(self, X):
        return np.sign(np.dot(X, self.w) + self.b)


In [ ]:

model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LinearSVC())
])

model.fit(X_train, y_train)


Pipeline(steps=[('tfidf', TfidfVectorizer(stop_words='english')),
                ('clf', LinearSVC())])

In [ ]:
# Ensure no missing values exist
print(X_train.isna().sum())

# If any appear:
X_train = X_train.fillna("")
X_test = X_test.fillna("")


0


In [ ]:
print(type(X_train))
print(X_train.iloc[0])


<class 'pandas.core.series.Series'>
Lions blow to World Cup winners  British and Irish Lions coach Clive Woodward says he is unlikely to select any players not involved in next year's RBS Six Nations Championship.  World Cup winners Lawrence Dallaglio, Neil Back and Martin Johnson had all been thought to be in the frame for next summer's tour to New Zealand. "I don't think you can ever say never," said Woodward. "But I would have to have a compulsive reason to pick any player who is not available to international rugby." Dallaglio, Back and Johnson have all retired from international rugby over the last 12 months but continue to star for their club sides. But Woodward added: "The key thing that I want to stress is that I intend to use the Six Nations and the players who are available to international rugby as the key benchmark. "My job, along with all the other senior representatives, is to make sure that we pick the strongest possible team. "If you are not playing international rugby 

In [ ]:
preds = model.predict(X_test)


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        84
           1       1.00      1.00      1.00       102

    accuracy                           1.00       186
   macro avg       1.00      1.00      1.00       186
weighted avg       1.00      1.00      1.00       186



In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5)
print(scores, scores.mean())


[1.         1.         0.99462366 1.         1.        ] 0.9989247311827956


Model 2: TF-IDF + Logistic Regression


In [ ]:
class LogisticRegression:
    def __init__(self, lr=0.01, epochs=1000):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = None

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.epochs):
            linear = np.dot(X, self.w) + self.b
            y_hat = self.sigmoid(linear)

            # gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_hat - y))
            db = (1 / n_samples) * np.sum(y_hat - y)

            # update
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        linear = np.dot(X, self.w) + self.b
        return self.sigmoid(linear)

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

model2 = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000))
])

model2.fit(X_train, y_train)


Pipeline(steps=[('tfidf', TfidfVectorizer(stop_words='english')),
                ('clf', LogisticRegression(max_iter=1000))])

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

preds2 = model2.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds2))
print(classification_report(y_test, preds2))


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        84
           1       1.00      1.00      1.00       102

    accuracy                           1.00       186
   macro avg       1.00      1.00      1.00       186
weighted avg       1.00      1.00      1.00       186



In [ ]:
from sklearn.model_selection import cross_val_score

scores_lr = cross_val_score(model2, X, y, cv=5)
print(scores_lr)
print(scores_lr.mean())


[0.98924731 1.         0.99462366 1.         0.99459459]
0.9956931124673061


In [ ]:
print("SVM CV Mean:", scores.mean())
print("LogReg CV Mean:", scores_lr.mean())


SVM CV Mean: 0.9989247311827956
LogReg CV Mean: 0.9956931124673061


Model 3: BoW + MNB


In [ ]:
class MultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.class_log_prior_ = None
        self.feature_log_prob_ = None
        self.classes_ = None

    def fit(self, X, y):
        """
        X: (n_samples, n_features) non-negative
        y: (n_samples,) class labels
        """
        n_samples, n_features = X.shape
        self.classes_, y_indices = np.unique(y, return_inverse=True)
        n_classes = len(self.classes_)

        # Class counts
        class_count = np.zeros(n_classes)
        feature_count = np.zeros((n_classes, n_features))

        for i in range(n_samples):
            class_count[y_indices[i]] += 1
            feature_count[y_indices[i]] += X[i]

        # Log priors
        self.class_log_prior_ = np.log(class_count / n_samples)

        # Log likelihoods with Laplace smoothing
        smoothed_fc = feature_count + self.alpha
        smoothed_cc = smoothed_fc.sum(axis=1, keepdims=True)

        self.feature_log_prob_ = np.log(smoothed_fc / smoothed_cc)

    def predict(self, X):
        log_probs = (
            X @ self.feature_log_prob_.T
            + self.class_log_prior_
        )
        return self.classes_[np.argmax(log_probs, axis=1)]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer

model3 = Pipeline([
    ("count", CountVectorizer(stop_words="english")),
    ("clf", MultinomialNB())
])

model3.fit(X_train, y_train)


Pipeline(steps=[('count', CountVectorizer(stop_words='english')),
                ('clf', MultinomialNB())])

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

preds2 = model3.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds2))
print(classification_report(y_test, preds2))


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        84
           1       1.00      1.00      1.00       102

    accuracy                           1.00       186
   macro avg       1.00      1.00      1.00       186
weighted avg       1.00      1.00      1.00       186



In [ ]:
from sklearn.model_selection import cross_val_score

scores_mlb = cross_val_score(model3, X, y, cv=5)
print(scores_mlb)
print(scores_mlb.mean())


[1.         1.         0.99462366 1.         0.99459459]
0.9978436501017146


In [ ]:
print("SVM CV Mean:", scores.mean())
print("LogReg CV Mean:", scores_mlb.mean())
print("MLB CV Mean:", scores_mlb.mean())


SVM CV Mean: 0.9989247311827956
LogReg CV Mean: 0.9978436501017146
MLB CV Mean: 0.9978436501017146
